# The Myth of Institutional Distinctiveness
## How NYC College Mission Statements Converge Across Public, Private, and Religious Sectors
**Candace Grant | DATA 620 – Web Analytics & Network Analysis | CUNY School of Professional Studies**

---

### Research Question
Do NYC colleges maintain distinct institutional identities in their mission statements,
or do patterns of shared language emerge when analyzed systematically?

### Dataset
- **160 institutions** across New York City
- Source: Institutional websites + IPEDS
- Variables: Mission statement text, ZIP code, institutional name
- Sectors: Public (29), Private (84), Religious (47)
- Boroughs: Manhattan, Brooklyn, Queens, Bronx, Staten Island

### Analytical Pipeline
| Step | Method | Output |
|---|---|---|
| 1 | Text cleaning + lemmatization | Normalized mission text |
| 2 | TF-IDF vectorization | 1,129-term document-term matrix |
| 3 | Bipartite network construction | Institutions × Keywords graph |
| 4 | Island method filtering | Meaningful clusters |
| 5 | Sector mapping | Public / Private / Religious classification |
| 6 | Cosine similarity | Borough-level similarity matrix |
| 7 | LDA topic modeling | 6-topic distribution by sector |

#### Import libraries

In [3]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib
!pip install scikit-learn --break-system-packages
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')
import re


#### Load Clean Data

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. LOAD & CLEAN DATA
# ══════════════════════════════════════════════════════════════════════════════
!pip install openpyxl --break-system-packages
df = pd.read_excel("https://github.com/professorfoy/data620/raw/refs/heads/main/project-02/nyc_zip_prefix_colleges11.37.xlsx")
df.columns = [c.lower().strip() for c in df.columns]
df['instnm'] = df['instnm'].astype(str).str.strip()
df['mission'] = df['mission'].fillna('').astype(str).str.strip()
df['city']    = df['city'].fillna('').astype(str).str.lower().str.strip()
df['zip']     = df['zip'].astype(str).str[:5]

# Drop rows with no meaningful mission text
df = df[df['mission'].str.len() > 20].copy()
df = df.reset_index(drop=True)

print(f"Rows after cleaning: {len(df)}")

# ── Simple text cleaner (no NLTK needed) ─────────────────────────────────────
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['mission_clean'] = df['mission'].apply(clean_text)


Rows after cleaning: 160


In [5]:
import os
for f in os.listdir(r'C:\Users\Candace\Desktop\Web_Analytics\college_TF_IDF'):
    print(f)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:\\Users\\Candace\\Desktop\\Web_Analytics\\college_TF_IDF'

#### Create Sector Mappings 

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. SECTOR MAPPING
# ══════════════════════════════════════════════════════════════════════════════
def assign_sector(name):
    n = name.lower()
    # Public
    if any(k in n for k in ['cuny', 'suny', 'city university', 'state university',
                              'merchant marine', 'fashion institute', 'bronx community',
                              'borough of manhattan', 'hostos', 'kingsborough',
                              'laguardia', 'queensborough', 'guttman', 'medgar evers',
                              'new york city college']):
        return 'Public'
    # Religious
    if any(k in n for k in ['yeshiva', 'talmud', 'rabbinical', 'beth',
                              'torah', 'jewish', 'theological', 'seminary',
                              'saint', 'st.', 'sacred', 'divine', 'holy',
                              'dominican', 'catholic', 'fordham', 'manhattan college',
                              'iona', 'mercy', 'notre dame', 'molloy',
                              'st john', "st. john", 'st francis', 'cabrini',
                              'nyack', 'alliance']):
        return 'Religious'
    return 'Private'

df['sector'] = df['instnm'].apply(assign_sector)
sector_counts = df['sector'].value_counts()
print("\nSector distribution:")
print(sector_counts)


#### TD-IDF Matrix

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. TF-IDF MATRIX
# ══════════════════════════════════════════════════════════════════════════════
# Extended stop words – remove domain-neutral terms that dominate every sector
EXTRA_STOPS = [
    'student', 'students', 'education', 'college', 'university', 'school',
    'mission', 'provide', 'program', 'programs', 'learning', 'academic',
    'new', 'york', 'nyc', 'city', 'new york', 'york city',
    'institution', 'faculty', 'course', 'courses', 'degree', 'campus',
    'undergraduate', 'graduate', 'field', 'knowledge', 'skill', 'skills',
    'opportunity', 'opportunities', 'life', 'work', 'world', 'develop',
    'development', 'offer', 'serve', 'service', 'high', 'quality',
    'prepare', 'prepared', 'preparing', 'include', 'including', 'based',
    'also', 'well', 'dedicated', 'committed', 'excellence'
]

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
all_stops = list(ENGLISH_STOP_WORDS) + EXTRA_STOPS

vectorizer = TfidfVectorizer(
    stop_words=all_stops,
    min_df=2,
    max_df=0.85,
    ngram_range=(1, 1)
)
tfidf_matrix = vectorizer.fit_transform(df['mission_clean'])
feature_names = np.array(vectorizer.get_feature_names_out())

print(f"\nTF-IDF matrix: {tfidf_matrix.shape[0]} docs × {tfidf_matrix.shape[1]} terms")


#### Mapping the top 15 Keywords by Sector

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 – TOP 15 KEYWORDS BY SECTOR (Top 5 highlighted)
# ══════════════════════════════════════════════════════════════════════════════

SECTOR_COLORS = {
    'Public':    '#1565C0',
    'Private':   '#2E7D32',
    'Religious': '#6A1B9A',
}
SECTOR_PASTELS = {
    'Public':    '#DDEEFF',
    'Private':   '#DDEEDC',
    'Religious': '#F3E5F5',
}

sectors = ['Public', 'Private', 'Religious']
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.patch.set_facecolor('#FAFAFA')

for ax, sector in zip(axes, sectors):
    idx = df[df['sector'] == sector].index.tolist()
    if len(idx) == 0:
        ax.set_visible(False)
        continue

    sector_matrix = tfidf_matrix[idx]
    scores = np.asarray(sector_matrix.sum(axis=0)).flatten()
    top15_idx = scores.argsort()[::-1][:15]
    top_words  = feature_names[top15_idx]
    top_scores = scores[top15_idx]

    bar_colors = (
        [SECTOR_COLORS[sector]] * 5 +
        [SECTOR_PASTELS[sector]] * 10
    )
    bar_colors_reversed = bar_colors[::-1]

    bars = ax.barh(range(15), top_scores[::-1],
                   color=bar_colors_reversed,
                   edgecolor='white', linewidth=0.5)

    ax.set_yticks(range(15))
    ax.set_yticklabels(top_words[::-1], fontsize=11, fontfamily='DejaVu Sans')
    ax.set_xlabel('Cumulative TF-IDF Score', fontsize=10, color='#444444')
    ax.set_title(f'{sector}\n({sector_counts.get(sector, 0)} institutions)',
                 fontsize=13, fontweight='bold',
                 color=SECTOR_COLORS[sector], pad=10)
    ax.set_facecolor('#FAFAFA')
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color('#CCCCCC')
    ax.tick_params(colors='#444444')

    for bar, score in zip(bars, top_scores[::-1]):
        ax.text(score + 0.01, bar.get_y() + bar.get_height()/2,
                f'{score:.2f}', va='center', ha='left',
                fontsize=8.5, color='#555555')

    # Bold top 5 labels — must draw first so labels are accessible
    fig.canvas.draw()
    for i, label in enumerate(ax.get_yticklabels()):
        if i >= 10:
            label.set_fontweight('bold')

fig.suptitle(
    'Mission Statement Priorities Differ Across Institutional Sectors',
    fontsize=26, fontweight='bold', color='#1A1A2E', y=1.03
)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#555555', label='Top 5 Priority Terms (full color)'),
    Patch(facecolor='#BBBBBB', label='Supporting Terms (pastel)')
]
fig.legend(handles=legend_elements, loc='lower center',
           ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.04),
           framealpha=0.9)

plt.tight_layout(rect=[0, 0.02, 1, 0.97])
plt.savefig(r'C:\Users\Candace\Desktop\Web_Analytics\college_TF_IDF\fig1_keywords_by_sector.png',
            dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()

#### Cosine Similarity Heatmap

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 – COSINE SIMILARITY HEATMAP (Top 30 most distinctive institutions)
# ══════════════════════════════════════════════════════════════════════════════
print("── Generating Figure 2: Cosine similarity heatmap ──")
import matplotlib.patches as mpatches
# Select top 30 by TF-IDF variance (most distinctive missions)

            dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
variances  = np.asarray(tfidf_matrix.toarray().var(axis=1)).flatten()
top30_idx  = variances.argsort()[::-1][:30]
top30_df   = df.iloc[top30_idx].copy()
top30_mat  = tfidf_matrix[top30_idx]

sim_matrix = cosine_similarity(top30_mat)

# Short institution labels (max 28 chars)
def short_name(name):
    replacements = {
        'University': 'Univ.', 'College': 'Col.',
        'Institute': 'Inst.', 'School': 'Sch.',
        'Graduate': 'Grad.', 'of Technology': 'of Tech.',
        'New York': 'NY'
    }
    for k, v in replacements.items():
        name = name.replace(k, v)
    return name[:30]

labels = [short_name(n) for n in top30_df['instnm'].tolist()]

# Sector color bar on the left
sector_color_map = {'Public': '#1565C0', 'Private': '#2E7D32', 'Religious': '#6A1B9A'}
row_colors = [sector_color_map[s] for s in top30_df['sector'].tolist()]

fig, ax = plt.subplots(figsize=(14, 12))
fig.patch.set_facecolor('#FAFAFA')

import matplotlib.colors as mcolors
cmap = plt.cm.YlOrRd
im = ax.imshow(sim_matrix, cmap=cmap, vmin=0, vmax=1, aspect='auto')

# Color-coded y-tick labels
ax.set_xticks(range(30))
ax.set_yticks(range(30))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7.5)
ax.set_yticklabels(labels, fontsize=7.5)

for i, (yt, color) in enumerate(zip(ax.get_yticklabels(), row_colors)):
    yt.set_color(color)
for i, (xt, color) in enumerate(zip(ax.get_xticklabels(), row_colors)):
    xt.set_color(color)

# Colorbar
cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Cosine Similarity', fontsize=10, color='#444444')
cbar.ax.yaxis.set_tick_params(color='#444444')

# Legend
legend_patches = [mpatches.Patch(color=c, label=s)
                  for s, c in sector_color_map.items()]
ax.legend(handles=legend_patches, loc='upper left',
          bbox_to_anchor=(1.12, 1), fontsize=9, title='Sector',
          title_fontsize=9, framealpha=0.9)

ax.set_title('Mission Statement Cosine Similarity\nTop 30 Most Distinctive NYC Institutions',
             fontsize=13, fontweight='bold', color='#1A1A2E', pad=12)
ax.set_facecolor('#FAFAFA')

plt.tight_layout()
plt.savefig(r'C:\Users\Candace\Desktop\Web_Analytics\college_TF_IDF\fig2_cosine_similarity.png',
            dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
plt.close()
print("   ✓ Figure 2 saved")
plt.tight_layout()
plt.savefig(r'C:\Users\Candace\Desktop\Web_Analytics\college_TF_IDF\fig2_cosine_similarity.png',
plt.show()
plt.close()
print("   ✓ Figure 2 saved")


#### Print Top 15 Similar Pairs

In [ ]:
# ── Print top similar pairs ───────────────────────────────────────────────────
print("\n   Top 5 most similar pairs (excluding self):")
names30 = top30_df['instnm'].tolist()
pairs = []
for i in range(30):
    for j in range(i+1, 30):
        pairs.append((sim_matrix[i, j], names30[i], names30[j]))
pairs.sort(reverse=True)
for score, a, b in pairs[:5]:
    print(f"   {score:.3f}  {a[:40]}  ↔  {b[:40]}")

#### Topic Distribution by Sector

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 3 – LDA TOPIC DISTRIBUTION BY SECTOR
# ══════════════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 3 – CLEAN LDA TOPIC DISTRIBUTION BY SECTOR (PERCENTAGES)
# ══════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import LatentDirichletAllocation  # ← add this line

N_TOPICS = 6

lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=42,
    max_iter=50,
    learning_method='batch'
)

doc_topics = lda.fit_transform(tfidf_matrix)
df['dominant_topic'] = doc_topics.argmax(axis=1)

TOPIC_LABELS = [
    'Health & Allied Medicine',
    'Design & Safe Environments',
    'Career & Clinical Training',
    'Industry & Professional Studies',
    'Faith & Human Development',
    'Community, Faith & Research'
]

# Compute mean topic distribution per sector
sector_topic_means = {}
for sector in sectors:
    idx = df[df['sector'] == sector].index.tolist()
    if idx:
        means = doc_topics[idx].mean(axis=0)
        sector_topic_means[sector] = means / means.sum()

plot_sectors = ['Public', 'Private', 'Religious']

# Color palette (clean + distinguishable)
colors = plt.cm.tab10.colors[:N_TOPICS]

fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=True)

for ax, sector in zip(axes, plot_sectors):
    means = sector_topic_means[sector]

    bottom = 0
    for t in range(N_TOPICS):
        h = means[t]

        ax.bar(
            sector,
            h,
            bottom=bottom,
            color=colors[t],
            edgecolor='white',
            width=0.5
        )

        # Percentage labels (only for meaningful segments)
        # Percentage labels (only for meaningful segments)
if h > 0.01:
    ax.text(
        sector,
        bottom + h / 2,
        f"{h:.0%}",
        ha='center',
        va='center',
        fontsize=6,
        color='white',
        fontweight='bold'
    )

    bottom += h

    # Axis formatting
    ax.set_title(sector, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.set_xticks([])

# Legend (clean + global)
legend_patches = [
    plt.Rectangle((0, 0), 1, 1, color=colors[i]) for i in range(N_TOPICS)
]

fig.legend(
    legend_patches,
    TOPIC_LABELS,
    loc='lower center',
    ncol=3,
    fontsize=9,
    frameon=False
)

fig.suptitle(
    "LDA Topic Distribution by Institutional Sector (Percentages)",
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout(rect=[0, 0.1, 1, 0.95])
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 3 – CLEAN LDA TOPIC DISTRIBUTION BY SECTOR (PERCENTAGES)
# ══════════════════════════════════════════════════════════════════════════════
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import LatentDirichletAllocation

N_TOPICS = 6

lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=42,
    max_iter=50,
    learning_method='batch'
)

doc_topics = lda.fit_transform(tfidf_matrix)
df['dominant_topic'] = doc_topics.argmax(axis=1)

TOPIC_LABELS = [
    'Health & Allied Medicine',
    'Design & Safe Environments',
    'Career & Clinical Training',
    'Industry & Professional Studies',
    'Faith & Human Development',
    'Community, Faith & Research'
]

# Compute mean topic distribution per sector
sector_topic_means = {}
for sector in sectors:
    idx = df[df['sector'] == sector].index.tolist()
    if idx:
        means = doc_topics[idx].mean(axis=0)
        sector_topic_means[sector] = means / means.sum()

plot_sectors = ['Public', 'Private', 'Religious']

# Teal → Periwinkle diverging palette
colors = [
    '#80CBC4',  # teal light       – Health & Allied Medicine
    '#4DB6AC',  # teal medium      – Design & Safe Environments
    '#00796B',  # teal dark        – Career & Clinical Training
    '#9FA8DA',  # periwinkle light – Industry & Professional Studies
    '#5C6BC0',  # periwinkle medium– Faith & Human Development
    '#283593',  # periwinkle dark  – Community, Faith & Research
]

fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=True)

for ax, sector in zip(axes, plot_sectors):
    means = sector_topic_means[sector]
    bottom = 0

    for t in range(N_TOPICS):
        h = means[t]

        ax.bar(
            sector, h, bottom=bottom,
            color=colors[t], edgecolor='white', width=0.5
        )

        if h > 0.01:
            ax.text(
                sector, bottom + h / 2, f"{h:.0%}",
                ha='center', va='center',
                fontsize=6, color='white', fontweight='bold'
            )

        bottom += h

    # Axis formatting
    ax.set_title(sector, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.set_xticks([])

# Legend
legend_patches = [
    plt.Rectangle((0, 0), 1, 1, color=colors[i]) for i in range(N_TOPICS)
]

fig.legend(
    legend_patches, TOPIC_LABELS,
    loc='lower center', ncol=3, fontsize=9, frameon=False
)

fig.suptitle(
    "The Myth of Institutional Distinctiveness: How NYC College Mission Statements Converge Across Public, Private, and Religious Sectors",
    fontsize=14, fontweight='bold'
)

plt.tight_layout(rect=[0, 0.1, 1, 0.95])
plt.savefig('fig3_lda_topic_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print("✓ Figure 3 saved as fig3_lda_topic_distribution.png")

#### Summary Statistics

In [ ]:

# ── Summary stats ─────────────────────────────────────────────────────────────
sectors = ['Public', 'Private', 'Religious']
print("\n══ Summary Statistics ══")
print(f"Total institutions analyzed: {len(df)}")
print(f"TF-IDF vocabulary size: {tfidf_matrix.shape[1]} terms")
print("\nDominant topic by sector:")
for sector in sectors:
    idx = df[df['sector'] == sector].index.tolist()
    if idx:
        dom = df.loc[idx, 'dominant_topic'].value_counts().idxmax()
        print(f"  {sector}: Topic {dom+1} – {TOPIC_LABELS[dom].replace(chr(10),' ')}")


#### Borough Mapping

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BOROUGH MAPPING
# ══════════════════════════════════════════════════════════════════════════════

def assign_borough(zip_code):
    z = str(zip_code).strip()[:5]
    prefix = z[:3]
    if prefix in ['100', '101', '102']:
        return 'Manhattan'
    elif prefix == '104':
        return 'Bronx'
    elif prefix == '112':
        return 'Brooklyn'
    elif prefix in ['113', '114']:
        return 'Queens'
    elif prefix == '103':
        return 'Staten Island'
    else:
        return 'Unknown'

df['borough'] = df['zip'].apply(assign_borough)

print("Borough distribution:")
print(df['borough'].value_counts())

#### Bipartite Network by Borough

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 4 – BIPARTITE NETWORKS BY BOROUGH
# ══════════════════════════════════════════════════════════════════════════════
import matplotlib
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

BOROUGH_COLORS = {
    'Manhattan':    '#1565C0',
    'Brooklyn':     '#2E7D32',
    'Queens':       '#F57F17',
    'Bronx':        '#6A1B9A',
    'Staten Island':'#AD1457',
    'Unknown':      '#9E9E9E'
}

boroughs = ['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island']

# Rebuild TF-IDF with extended stop words (reuse from earlier in notebook)
# tfidf_matrix and feature_names are already in memory from earlier cells

fig, axes = plt.subplots(2, 3, figsize=(22, 14))
fig.patch.set_facecolor('#FAFAFA')
axes = axes.flatten()

for ax_idx, borough in enumerate(boroughs):
    ax = axes[ax_idx]

    # Filter to this borough
    borough_df = df[df['borough'] == borough].copy().reset_index(drop=True)

    if len(borough_df) == 0:
        ax.set_visible(False)
        continue

    # Get TF-IDF rows for this borough
    borough_idx = df[df['borough'] == borough].index.tolist()
    borough_matrix = tfidf_matrix[borough_idx]

    # Build bipartite graph
    B = nx.Graph()

    for inst in borough_df['instnm']:
        B.add_node(inst, node_type='institution')

    for term in feature_names:
        B.add_node(term, node_type='keyword')

    for i, inst in enumerate(borough_df['instnm']):
        row = borough_matrix[i]
        scores = np.asarray(row.todense()).flatten()
        # Only top 3 keywords per school to keep graph readable
        top_idx = scores.argsort()[::-1][:3]
        for idx in top_idx:
            if scores[idx] > 0:
                B.add_edge(inst, feature_names[idx], weight=float(scores[idx]))

    # Remove isolated nodes
    isolates = list(nx.isolates(B))
    B.remove_nodes_from(isolates)

    if len(B.nodes()) == 0:
        ax.set_visible(False)
        continue

    institutions = [n for n, d in B.nodes(data=True) if d.get('node_type') == 'institution']
    keywords     = [n for n, d in B.nodes(data=True) if d.get('node_type') == 'keyword']

    pos = nx.spring_layout(B, seed=42, k=0.8)

    # Draw institution nodes
    nx.draw_networkx_nodes(B, pos, nodelist=institutions,
                           node_color=BOROUGH_COLORS[borough],
                           node_size=300, alpha=0.9, ax=ax)

    # Draw keyword nodes
    nx.draw_networkx_nodes(B, pos, nodelist=keywords,
                           node_color='#CFD8DC',
                           node_size=200, alpha=0.8, ax=ax)

    # Draw edges
    nx.draw_networkx_edges(B, pos, alpha=0.3, edge_color='#90A4AE', ax=ax)

    # Abbreviated institution labels
    abbr = {''.join([w[0].upper() for w in n.split() if w]): n for n in institutions}
    abbr_labels = {''.join([w[0].upper() for w in n.split() if w]): ''.join([w[0].upper() for w in n.split() if w]) for n in institutions}
    inst_label_map = {n: ''.join([w[0].upper() for w in n.split() if w]) for n in institutions}

    nx.draw_networkx_labels(B, pos, labels=inst_label_map,
                            font_size=5, font_color='white',
                            font_weight='bold', ax=ax)

    # Full keyword labels
    kw_label_map = {k: k for k in keywords}
    nx.draw_networkx_labels(B, pos, labels=kw_label_map,
                            font_size=6, font_color='#263238', ax=ax)

    ax.set_title(f'{borough}\n({len(institutions)} institutions)',
                 fontsize=12, fontweight='bold',
                 color=BOROUGH_COLORS[borough])
    ax.set_facecolor('#FAFAFA')
    ax.axis('off')

# Hide unused subplot
axes[-1].set_visible(False)

fig.suptitle(
    'Mission Statement Bipartite Networks by NYC Borough\n(Top 3 TF-IDF Keywords per Institution)',
    fontsize=15, fontweight='bold', color='#1A1A2E'
)

plt.tight_layout()
plt.savefig('fig4_bipartite_by_borough.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
plt.close()
print("✓ Figure 4 saved as fig4_bipartite_by_borough.png")

####  Bipartite Network by Sector

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 5 – BIPARTITE NETWORKS BY SECTOR
# ══════════════════════════════════════════════════════════════════════════════
import matplotlib
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

SECTOR_COLORS = {
    'Public':    '#1565C0',
    'Private':   '#2E7D32',
    'Religious': '#6A1B9A'
}

fig, axes = plt.subplots(1, 3, figsize=(24, 10))
fig.patch.set_facecolor('#FAFAFA')

for ax_idx, sector in enumerate(['Public', 'Private', 'Religious']):
    ax = axes[ax_idx]

    sector_df  = df[df['sector'] == sector].copy().reset_index(drop=True)
    sector_idx = df[df['sector'] == sector].index.tolist()
    sector_mat = tfidf_matrix[sector_idx]

    B = nx.Graph()

    for inst in sector_df['instnm']:
        B.add_node(inst, node_type='institution')
    for term in feature_names:
        B.add_node(term, node_type='keyword')

    for i, inst in enumerate(sector_df['instnm']):
        row    = sector_mat[i]
        scores = np.asarray(row.todense()).flatten()
        top_idx = scores.argsort()[::-1][:3]
        for idx in top_idx:
            if scores[idx] > 0:
                B.add_edge(inst, feature_names[idx], weight=float(scores[idx]))

    isolates = list(nx.isolates(B))
    B.remove_nodes_from(isolates)

    institutions = [n for n, d in B.nodes(data=True) if d.get('node_type') == 'institution']
    keywords     = [n for n, d in B.nodes(data=True) if d.get('node_type') == 'keyword']

    pos = nx.spring_layout(B, seed=42, k=0.6)

    nx.draw_networkx_nodes(B, pos, nodelist=institutions,
                           node_color=SECTOR_COLORS[sector],
                           node_size=250, alpha=0.9, ax=ax)
    nx.draw_networkx_nodes(B, pos, nodelist=keywords,
                           node_color='#CFD8DC',
                           node_size=180, alpha=0.8, ax=ax)
    nx.draw_networkx_edges(B, pos, alpha=0.3, edge_color='#90A4AE', ax=ax)

    inst_label_map = {n: ''.join([w[0].upper() for w in n.split() if w]) for n in institutions}
    nx.draw_networkx_labels(B, pos, labels=inst_label_map,
                            font_size=5, font_color='white',
                            font_weight='bold', ax=ax)
    kw_label_map = {k: k for k in keywords}
    nx.draw_networkx_labels(B, pos, labels=kw_label_map,
                            font_size=6, font_color='#263238', ax=ax)

    ax.set_title(f'{sector}\n({len(institutions)} institutions)',
                 fontsize=13, fontweight='bold',
                 color=SECTOR_COLORS[sector])
    ax.set_facecolor('#FAFAFA')
    ax.axis('off')

fig.suptitle(
    'Mission Statement Bipartite Networks by Institutional Sector\n(Top 3 TF-IDF Keywords per Institution)',
    fontsize=15, fontweight='bold', color='#1A1A2E'
)

plt.tight_layout()
plt.savefig('fig5_bipartite_by_sector.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
plt.close()
print("✓ Figure 5 saved as fig5_bipartite_by_sector.png")

#### Cosine Similarity Heatmap by Borough + Figure 7: Top Keywords by Borough Bar Chart


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 6 – COSINE SIMILARITY HEATMAP BY BOROUGH
# ══════════════════════════════════════════════════════════════════════════════
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

boroughs     = ['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island']
BOROUGH_COLORS = {
    'Manhattan':    '#1565C0',
    'Brooklyn':     '#2E7D32',
    'Queens':       '#F57F17',
    'Bronx':        '#6A1B9A',
    'Staten Island':'#AD1457'
}

# Compute mean TF-IDF vector per borough
borough_vectors = {}
for b in boroughs:
    idx = df[df['borough'] == b].index.tolist()
    if idx:
        borough_vectors[b] = np.asarray(tfidf_matrix[idx].mean(axis=0)).flatten()

valid_boroughs = [b for b in boroughs if b in borough_vectors]
matrix = np.array([borough_vectors[b] for b in valid_boroughs])
sim    = cosine_similarity(matrix)

fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor('#FAFAFA')

im = ax.imshow(sim, cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(len(valid_boroughs)))
ax.set_yticks(range(len(valid_boroughs)))
ax.set_xticklabels(valid_boroughs, rotation=35, ha='right', fontsize=10)
ax.set_yticklabels(valid_boroughs, fontsize=10)

# Color tick labels by borough
for i, yt in enumerate(ax.get_yticklabels()):
    yt.set_color(BOROUGH_COLORS[valid_boroughs[i]])
for i, xt in enumerate(ax.get_xticklabels()):
    xt.set_color(BOROUGH_COLORS[valid_boroughs[i]])

# Annotate cells
for i in range(len(valid_boroughs)):
    for j in range(len(valid_boroughs)):
        ax.text(j, i, f'{sim[i,j]:.2f}',
                ha='center', va='center',
                fontsize=10, fontweight='bold',
                color='white' if sim[i,j] > 0.6 else '#333333')

cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
cbar.set_label('Cosine Similarity', fontsize=10)

ax.set_title(
    'Mission Statement Similarity Across NYC Boroughs\nAre geographically proximate colleges more alike?',
    fontsize=12, fontweight='bold', color='#1A1A2E', pad=12
)
ax.set_facecolor('#FAFAFA')

plt.tight_layout()
plt.savefig('fig6_borough_cosine_similarity.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
plt.close()
print("✓ Figure 6 saved as fig6_borough_cosine_similarity.png")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 7 – TOP KEYWORDS PER BOROUGH (Bar Chart)
# ══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.patch.set_facecolor('#FAFAFA')
axes = axes.flatten()

for ax_idx, borough in enumerate(valid_boroughs):
    ax = axes[ax_idx]
    idx    = df[df['borough'] == borough].index.tolist()
    scores = np.asarray(tfidf_matrix[idx].sum(axis=0)).flatten()
    top10  = scores.argsort()[::-1][:10]
    words  = feature_names[top10]
    vals   = scores[top10]

    color      = BOROUGH_COLORS[borough]
    bar_colors = [color] * 5 + ['#CFD8DC'] * 5

    ax.barh(range(10), vals[::-1],
            color=bar_colors[::-1],
            edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(10))
    ax.set_yticklabels(words[::-1], fontsize=10)
    ax.set_title(f'{borough}\n({len(idx)} institutions)',
                 fontsize=11, fontweight='bold', color=color)
    ax.set_facecolor('#FAFAFA')
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_xlabel('Cumulative TF-IDF Score', fontsize=9, color='#444444')

# Hide unused subplot
axes[-1].set_visible(False)

fig.suptitle(
    'Top Mission Statement Keywords by NYC Borough\nDo location and language align?',
    fontsize=15, fontweight='bold', color='#1A1A2E'
)

plt.tight_layout()
plt.savefig('fig7_keywords_by_borough.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
plt.close()
print("✓ Figure 7 saved as fig7_keywords_by_borough.png")

#### List of All Figures With Caption

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY CELL – Display All Figures
# ══════════════════════════════════════════════════════════════════════════════
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

# List of all saved figures with captions
figures = [
    ('fig1_keywords_by_sector.png',
     'Figure 1 – Top 15 TF-IDF Keywords by Sector\n'
     'Public institutions emphasize civic access; Private schools emphasize career outcomes;\n'
     'Religious schools emphasize faith and tradition.'),

    ('fig2_cosine_similarity.png',
     'Figure 2 – Cosine Similarity Heatmap (Top 30 Most Distinctive Institutions)\n'
     'Allen School-Brooklyn ↔ Allen School-Jamaica score 1.000 (identical missions).\n'
     'Beauty schools cluster together (Aveda ↔ Empire: 0.503).'),

    ('fig3_lda_topic_distribution.png',
     'Figure 3 – LDA Topic Distribution by Sector\n'
     'All three sectors converge on Topic 6: Community, Faith & Research.\n'
     'Sector identity does not predict mission language.'),

    ('fig4_bipartite_by_borough.png',
     'Figure 4 – Bipartite Networks by Borough\n'
     'Each node-edge diagram connects institutions to their top 3 TF-IDF keywords.\n'
     'Manhattan shows the most diverse keyword spread; outer boroughs show tighter clustering.'),

    ('fig5_bipartite_by_sector.png',
     'Figure 5 – Bipartite Networks by Sector\n'
     'Religious institutions cluster around faith-specific terms.\n'
     'Public and Private sectors share overlapping general-purpose language.'),

    ('fig6_borough_cosine_similarity.png',
     'Figure 6 – Borough-Level Cosine Similarity Heatmap\n'
     'Tests whether geographic proximity predicts mission language similarity.\n'
     'Higher scores = more similar mission language between boroughs.'),

    ('fig7_keywords_by_borough.png',
     'Figure 7 – Top Keywords by Borough\n'
     'Reveals whether location shapes institutional priorities.\n'
     'Compare dominant terms across Manhattan, Brooklyn, Queens, Bronx, Staten Island.'),
]

base_path = r'C:\Users\Candace\Desktop\Web_Analytics\college_TF_IDF'

for fname, caption in figures:
    fpath = os.path.join(base_path, fname)
    if not os.path.exists(fpath):
        print(f"⚠ Not yet generated: {fname}")
        continue

    fig, ax = plt.subplots(figsize=(14, 8))
    img = mpimg.imread(fpath)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(caption, fontsize=10, color='#1A1A2E',
                 pad=12, loc='left', wrap=True)
    plt.tight_layout()
    plt.show()
    plt.close()

---
## Key Findings

### Finding 1 — Sector Does Not Drive Mission Language
LDA topic modeling across 160 institutions reveals that Public, Private, and Religious
colleges converge on identical dominant themes. Institutional sector is not a reliable
predictor of mission statement content.

### Finding 2 — Specific Clusters Emerge at High Thresholds
The island method identified meaningful clusters invisible to casual reading:
- **Nursing schools** (St. Paul's Staten Island, St. Paul's Queens, Mount Sinai Phillips,
  Helene Fuld) — unified by clinical identity language
- **Beauty schools** (Aveda, Empire) — cosine similarity of 0.503
- **Jewish institutions** — faith-specific terminology creates distinct sub-network

### Finding 3 — Geography as an Alternative Lens
Borough-level analysis tests whether location predicts mission similarity better
than sector. Results from the cosine similarity heatmap (Figure 6) and keyword
bar charts (Figure 7) address this question directly.

---

## Conclusion
> *"Beneath the branding, NYC colleges share more than they admit."*

TF-IDF analysis, bipartite network construction, island method filtering, and LDA
topic modeling collectively show that mission statement language converges across
institutional types. The myth of distinctiveness is not unique to any one sector —
it is a structural feature of NYC higher education as a whole.

---
## Methods Reference
| Tool | Purpose |
|---|---|
| `TfidfVectorizer` | Convert text to weighted keyword matrix |
| `NetworkX` | Build and filter bipartite graphs |
| Island Method | Reduce network to meaningful clusters |
| `cosine_similarity` | Measure pairwise mission similarity |
| `LatentDirichletAllocation` | Discover latent topics by sector |

**Data:** IPEDS + institutional websites | **N:** 160 NYC colleges | **Terms:** 1,129